# A four-bar linkage

Four links joined by four hinges in a loop: a crank turning about one ground pivot, a coupler, and a rocker that swings about a second ground pivot. The engine builds it as a tree of three hinged bodies, with a closure joining the rocker's far end to the ground, and keeps that loop closed as it moves. The notebook checks:

1. the angles of the links over a full turn of the crank, and their rates, against the geometry of the loop;
2. the two places where the rocker stops and turns back, from the law of cosines;
3. the rest position on a spring, where the potential energy is least;
4. the frequency of small swings about that rest, from Lagrange's equation with the closed-form geometry.

Then it swings the linkage and plays the swing back in Rerun.

Each check prints a result beside its closed form and stops the notebook if they differ by more than the allowance derived beside it. `python/tests/test_notebooks.py` runs the notebook in the same way, so these numbers are tested. (Plan task 6.6.)

In [ ]:
import cmath
import math
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import multibody as mbd

# The loop is planar, but its closure is a spatial hinge: three of its five equations repeat the other
# two, and every solve says so (MBD-K050). Expected here, so not shown.
warnings.filterwarnings("ignore", message="MBD-K050", category=mbd.MbdWarning)


def check(what, value, expected, allowed):
    """Print a result beside the value it should have; stop if they differ by more than `allowed`."""
    difference = float(np.max(np.abs(np.asarray(value) - np.asarray(expected))))
    if np.ndim(value) == 0:
        print(f"{what}: {value:.12g}, expected {expected:.12g}, difference {difference:.1e} (allowed {allowed:.1e})")
    else:
        print(f"{what}: largest difference {difference:.1e} (allowed {allowed:.1e})")
    assert difference <= allowed, what

## The model

The crank is a = 0.3 m long, the coupler b = 0.8 m and the rocker c = 0.6 m, and the ground pivots are d = 0.7 m apart: O at the origin and D at (d, 0). The links are uniform bars 4 cm square, of 1, 2 and 1.5 kg. The hinges turn about Z, and gravity is −Y. A spring of 300 N/m with a free length of 0.2 m hangs the coupler from the point (0.35, 0.9), pulling at 0.4 m along it.

Each body's frame is at its hinge with the previous link, its X axis along its own link. So the coordinates are the angles at the hinges: q0 is the crank's angle from X, q1 the coupler's from the crank, and q2 the rocker's from the coupler. The rocker's X axis points from its hinge with the coupler, C, to the ground pivot D, where the closure holds its far end.

The shortest and the longest links together, 0.3 + 0.8 = 1.1 m, are shorter than the other two, 0.6 + 0.7 = 1.3 m. By Grashof's rule the shortest link, here the crank beside the ground, can then turn fully, while the rocker rocks.

In [ ]:
a, b, c, d = 0.3, 0.8, 0.6, 0.7
lengths = {"crank": a, "coupler": b, "rocker": c}
masses = {"crank": 1.0, "coupler": 2.0, "rocker": 1.5}
anchor, spring_at, stiffness, free_length = (0.35, 0.9), 0.4, 300.0, 0.2
g = mbd.g_accel


def bar(name):
    """A uniform bar along its body's X axis from the hinge, 4 cm square."""
    inertia = mbd.RigidBodyInertia.from_solid_box(masses[name], [0.5 * lengths[name], 0.02, 0.02])
    inertia.com_B = [0.5 * lengths[name], 0.0, 0.0]
    return inertia


def four_bar(crank_driver=None):
    """The linkage on its spring; given a TimeFunction, a driver turns the crank along it."""
    system = mbd.System()
    model = system.model
    model.gravity = [0.0, -g, 0.0]
    hinge = mbd.RevoluteJointModel()
    model.add_body("ground", hinge, inertia=bar("crank"), name="crank")
    model.add_body("crank", hinge, x_pj=[a, 0.0, 0.0], inertia=bar("coupler"), name="coupler")
    model.add_body("coupler", hinge, x_pj=[b, 0.0, 0.0], inertia=bar("rocker"), name="rocker")
    system.add_constraint(mbd.revolute_closure(mbd.Marker(0, [d, 0.0, 0.0]),
                                               mbd.Marker(model.body("rocker"), [c, 0.0, 0.0])))
    system.add_force_element(mbd.LinearSpringDamper(0, model.body("coupler"), [*anchor, 0.0],
                                                    [spring_at, 0.0, 0.0], stiffness, 0.0, free_length))
    if crank_driver is not None:
        system.add_constraint(mbd.JointDriver(model, model.body("crank"), crank_driver))
    return system

## The loop's geometry

Given the crank angle θ, the crank's end B is at a e^{iθ} in the complex plane. C lies both on a circle of radius b about B and on one of radius c about D. The two circles meet twice: this linkage is assembled with C to the left of the line from B to D, above it. From B, C and D come the absolute angles of the links, φ1 = θ, φ2 of BC and φ3 of CD, and the joint coordinates q = (φ1, φ2 − φ1, φ3 − φ2).

In [ ]:
def link_angles(theta):
    """The links' absolute angles (phi1, phi2, phi3): crank, coupler from B to C, rocker from C to D."""
    B = a * cmath.exp(1j * theta)
    BD = abs(d - B)
    along = (b * b - c * c + BD * BD) / (2.0 * BD)           # from B towards D, to the foot of C
    C = B + (d - B) / BD * complex(along, math.sqrt(b * b - along * along))   # to the left of BD
    return theta, cmath.phase(C - B), cmath.phase(d - C)


def coordinates(theta):
    phi1, phi2, phi3 = link_angles(theta)
    return np.array([phi1, phi2 - phi1, phi3 - phi2])


def wrapped(angle):
    """An angle, or angles, brought into [-pi, pi)."""
    return (angle + math.pi) % (2.0 * math.pi) - math.pi


report = mbd.validate(four_bar(), coordinates(1.0))
print(f"{report.constraint_equations} closure equations, {report.independent_equations} independent; "
      f"{report.degrees_of_freedom} degree of freedom")

## A full turn of the crank

A driver turns the crank, its angle following s(t) = t, so the driving parameter t is the crank angle itself. With the crank given nothing is left free, and `mbd.Kinematics` solves the closure for the other two angles at each t, starting from the previous solution so that it stays on the same branch. `velocities()` gives the rates dq/dt, which with ds/dt = 1 are the rates per radian of crank.

Differentiating the loop, a e^{iφ1} + b e^{iφ2} + c e^{iφ3} = d, gives the rates in closed form:

$$\dot\varphi_2 = -\frac{a \sin(\varphi_1 - \varphi_3)}{b \sin(\varphi_2 - \varphi_3)}\,\dot\varphi_1, \qquad \dot\varphi_3 = -\frac{a \sin(\varphi_1 - \varphi_2)}{c \sin(\varphi_3 - \varphi_2)}\,\dot\varphi_1.$$

The solve stops at a residual of 1e-10 m, and its Newton steps converge quadratically, so the angles are far closer than that: allowed 1e-9 rad, and 1e-9 for the rates.

In [ ]:
kinematics = mbd.Kinematics(four_bar(mbd.TimeFunction(lambda t: t, lambda t: 1.0, lambda t: 0.0)))
crank = np.linspace(0.0, 2.0 * math.pi, 361)
q_turn, rates = [], []
kinematics.q[:] = coordinates(crank[0])
for theta in crank:
    kinematics.t = theta
    assert kinematics.solve()
    q_turn.append(kinematics.q.copy())
    rates.append(kinematics.velocities())
q_turn, rates = np.array(q_turn), np.array(rates)

check("joint angles over the turn [rad]", wrapped(q_turn - [coordinates(theta) for theta in crank]), 0.0, 1e-9)
expected = []
for theta in crank:
    phi1, phi2, phi3 = link_angles(theta)
    coupler_rate = -a * math.sin(phi1 - phi3) / (b * math.sin(phi2 - phi3))
    rocker_rate = -a * math.sin(phi1 - phi2) / (c * math.sin(phi3 - phi2))
    expected.append([1.0, coupler_rate - 1.0, rocker_rate - coupler_rate])
check("their rates per radian of crank", rates, expected, 1e-9)

## Where the rocker turns back

The rocker stops where crank and coupler are in line, stretched out or folded. C is then b + a or b − a from O, and the triangle O, C, D, with its other sides c and d, fixes C by the law of cosines. At those two crank angles the rocker's rate must be zero, and its angle that of CD. Each solve starts from the nearest point of the turn above; allowed 1e-9 as before.

In [ ]:
turning_points = []
for reach in (b + a, b - a):                               # stretched out, then folded
    x = (reach**2 - c**2 + d**2) / (2.0 * d)                # C's distance along OD: the law of cosines
    C = complex(x, math.sqrt(reach**2 - x**2))              # above the ground line, as assembled
    theta = cmath.phase(C) % (2.0 * math.pi)
    if reach < b:
        theta = (theta + math.pi) % (2.0 * math.pi)         # folded: the crank points away from C
    kinematics.q[:] = q_turn[np.argmin(np.abs(crank - theta))]
    kinematics.t = theta
    assert kinematics.solve()
    rocker = cmath.phase(d - C)
    check(f"rocker's angle at a crank angle of {theta:.4f} rad [rad]", wrapped(np.sum(kinematics.q) - rocker), 0.0, 1e-9)
    check("  its rate [rad per rad]", np.sum(kinematics.velocities()), 0.0, 1e-9)
    turning_points.append((theta, rocker))

fig, (top, bottom) = plt.subplots(2, 1, figsize=(8, 5.5), sharex=True)
top.plot(np.degrees(crank), np.degrees(wrapped(q_turn[:, 0] + q_turn[:, 1])), label=r"coupler, $\varphi_2$")
top.plot(np.degrees(crank), np.degrees(wrapped(np.sum(q_turn, axis=1))), label=r"rocker, $\varphi_3$")
top.plot(*np.degrees(np.array(turning_points)).T, "ko", label="the rocker turns back")
top.set_ylabel("absolute angle [°]")
top.legend()
top.grid(True)
bottom.plot(np.degrees(crank), rates[:, 0] + rates[:, 1], label="coupler")
bottom.plot(np.degrees(crank), np.sum(rates, axis=1), label="rocker")
bottom.set_xlabel("crank angle θ [°]")
bottom.set_ylabel("rate per rate of crank")
bottom.legend()
bottom.grid(True)

## Resting on the spring

`static_equilibrium` moves the linkage, from the crank at 1.2 rad, to where gravity and the spring balance. There the potential energy V(θ), gravity's and the spring's, is stationary along the one freedom left. Below, V is written from the closed-form geometry, with each bar's centre at its middle and the spring's end 0.4 m along the coupler. Its derivative, by central differences of step 1e-4 rad, errs by about h² V‴ / 6, some 1e-8 N m, and statics stops when the acceleration is below 1e-6 rad/s², which for this linkage's generalised mass of 0.27 kg m² (below) is a torque below 2.7e-7 N m: allowed 1e-6 N m.

In [ ]:
def points(theta):
    """At crank angle theta: the links' absolute angles, their centres and the spring's end (complex numbers)."""
    phi1, phi2, phi3 = link_angles(theta)
    B = a * cmath.exp(1j * phi1)
    C = B + b * cmath.exp(1j * phi2)
    angles = {"crank": phi1, "coupler": phi2, "rocker": phi3}
    centres = {"crank": B / 2.0, "coupler": (B + C) / 2.0, "rocker": (C + d) / 2.0}
    return angles, centres, B + spring_at * cmath.exp(1j * phi2)


def potential(theta):
    _, centres, end = points(theta)
    stretch = abs(end - complex(*anchor)) - free_length
    return sum(masses[n] * g * centres[n].imag for n in masses) + 0.5 * stiffness * stretch**2


sim = mbd.Simulator(four_bar())
sim.q[:] = coordinates(1.2)
print(mbd.static_equilibrium(sim))
rest = sim.q[0]
check("the loop closed at rest [rad]", wrapped(sim.q - coordinates(rest)), 0.0, 1e-9)
step = 1e-4
check("dV/dtheta at rest [N m]", (potential(rest + step) - potential(rest - step)) / (2.0 * step), 0.0, 1e-6)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(np.degrees(crank), [potential(theta) for theta in crank], label="V(θ), gravity and spring")
ax.plot(np.degrees(rest), potential(rest), "ko", label="at rest")
ax.set_xlabel("crank angle θ [°]")
ax.set_ylabel("[J]")
ax.legend()
ax.grid(True)

## Small swings about the rest

With one freedom the kinetic energy is ½ M(θ) θ̇², where the generalised mass is

$$M(\theta) = \sum_i m_i \left|\frac{d\mathbf r_i}{d\theta}\right|^2 + I_i \left(\frac{d\varphi_i}{d\theta}\right)^2,$$

with r_i the centre of each bar and I_i its moment about its centre, m (ℓ²/4 + 0.02²) / 3 for a box of half-extents ℓ/2 and 0.02 m. Lagrange's equation about the minimum of V gives small swings at

$$\omega = \sqrt{V''(\theta^*) / M(\theta^*)}.$$

V″ comes from central differences of step 1e-3: their truncation, about h² V⁗ / 12, is some 1e-7 of V″, and their rounding, about 4 × 1e-16 × V / h², some 1e-8. M comes from central differences of step 1e-6, with a rounding error near 1e-10. ω carries half of V″'s relative error: allowed 1e-6 relative. `linearize` differentiates the engine's own equations instead, on the constraints.

In [ ]:
def generalised_mass(theta, step=1e-6):
    angles_ahead, centres_ahead, _ = points(theta + step)
    angles_behind, centres_behind, _ = points(theta - step)
    moment = {n: masses[n] * ((0.5 * lengths[n]) ** 2 + 0.02**2) / 3.0 for n in masses}
    return sum(masses[n] * abs((centres_ahead[n] - centres_behind[n]) / (2.0 * step)) ** 2
               + moment[n] * (wrapped(angles_ahead[n] - angles_behind[n]) / (2.0 * step)) ** 2 for n in masses)


step = 1e-3
curvature = (potential(rest + step) - 2.0 * potential(rest) + potential(rest - step)) / step**2
omega = math.sqrt(curvature / generalised_mass(rest))
linear = mbd.linearize(sim)
print(linear)
check("frequency of small swings [rad/s]", linear.modes[0].natural_frequency, omega, 1e-6 * omega)

## A swing, played back

From rest the crank is set turning at 4 rad/s; `assemble`, with the crank's joint held, makes the other rates agree with it, since the loop allows only one. Dormand-Prince runs it for 3 s. Nothing here dissipates energy, so the kinetic energy (`mbd.kinetic_energy`), gravity's (`mbd.potential_energy`) and the spring's, evaluated at each recorded state, add up to a constant up to the integrator's error, which is printed below. Not enough to turn the crank over: the linkage swings back and forth about its rest.

`mbd.replay` then plays the swing in Rerun, with the energies plotted beside it.

In [ ]:
sim.v[0] = 4.0
mbd.assemble(sim, mbd.AssemblyOptions().hold_joint(sim.system.model, 1))
sim.method = mbd.Integrator.DormandPrince
sim.initialize()
run = sim.run_recorded(3.0, 1e-3)

probe = mbd.Simulator(four_bar())                    # evaluates the energies at each recorded state
coupler = probe.system.model.body("coupler")


def energies(q, v):
    probe.q[:], probe.v[:] = q, v
    probe.refresh()
    end = probe.states()[coupler].pose_WB().apply([spring_at, 0.0, 0.0])
    stretch = math.hypot(end[0] - anchor[0], end[1] - anchor[1]) - free_length
    return mbd.kinetic_energy(probe), mbd.potential_energy(probe), 0.5 * stiffness * stretch**2


energy = pd.DataFrame([energies(q, v) for q, v in zip(run["q"], run["v"])],
                      columns=["kinetic", "gravity", "spring"], index=pd.Index(run["time"], name="time"))
energy["total"] = energy.sum(axis=1)
print(f"total energy {energy['total'].iloc[0]:.6f} J, changing by at most {np.ptp(energy['total']):.1e} J")
ax = energy.plot(figsize=(8, 3.5), grid=True, ylabel="energy [J]", xlabel="time [s]")
ax.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0));

In [ ]:
mbd.replay(sim.system, run, name="four-bar", channels=energy);